# FINPATH AI — Master Build & ML Training Notebook (Google Colab)
**Production-Ready Fintech Transaction Intelligence Dashboard & Dual-Model Pipeline**

> **Important Note on Colab Runtime**: Google Colab is an ephemeral notebook runtime used here to generate the 200,000 synthetic transaction dataset, train and evaluate both Scikit-Learn Random Forest models, save the `.joblib` model binaries and evaluation JSON artifacts, and package the complete full-stack application into `finpath-ai.zip` for local Docker execution or permanent cloud hosting.

In [ ]:
# STEP 1: Install Required Dependencies
!pip install -q numpy pandas scikit-learn joblib fastapi uvicorn sqlalchemy pydantic pydantic-settings email-validator "python-jose[cryptography]" bcrypt python-multipart

In [ ]:
# STEP 2, 3, 4 & 5: Generate 200,000 Transactions, Train Model 1 & Model 2, Evaluate, and Save Artifacts
import os
import sys
import json
from pathlib import Path

# Ensure project root is on sys.path
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.train import train_and_save_all

model1, model2, df, eval_metrics, feature_meta, analytics = train_and_save_all(
    artifacts_dir=PROJECT_ROOT / 'ml' / 'artifacts',
    n_samples=200_000,
    seed=42
)

print('\n=== MODEL 1: TRANSACTION SUCCESS PREDICTOR ===')
print(json.dumps({
    'accuracy': eval_metrics['model1']['accuracy'],
    'precision_macro': eval_metrics['model1']['precision_macro'],
    'recall_macro': eval_metrics['model1']['recall_macro'],
    'f1_macro': eval_metrics['model1']['f1_macro'],
    'confusion_matrix': eval_metrics['model1']['confusion_matrix']
}, indent=2))

print('\n=== MODEL 2: FAILURE NODE ANALYZER ===')
print(json.dumps({
    'accuracy': eval_metrics['model2']['accuracy'],
    'precision_macro': eval_metrics['model2']['precision_macro'],
    'recall_macro': eval_metrics['model2']['recall_macro'],
    'f1_macro': eval_metrics['model2']['f1_macro'],
    'confusion_matrix': eval_metrics['model2']['confusion_matrix']
}, indent=2))

In [ ]:
# STEP 6, 7, 8 & 9: Package Complete Project + Trained Artifacts into finpath-ai.zip and Download
import zipfile

zip_filename = 'finpath-ai.zip'
exclude_dirs = {'.git', 'node_modules', 'dist', '__pycache__', '.venv', 'venv', '.ipynb_checkpoints'}

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk('.'):
        dirs[:] = [d for d in dirs if d not in exclude_dirs]
        for file in files:
            if file == zip_filename or file.endswith('.pyc'):
                continue
            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, '.')
            zf.write(full_path, os.path.join('finpath-ai', rel_path))

print(f'Successfully packaged {zip_filename} ({os.path.getsize(zip_filename) / 1024:.1f} KB)')

try:
    from google.colab import files
    files.download(zip_filename)
except ImportError:
    print('Not running inside Google Colab browser session; finpath-ai.zip is saved in the working directory.')

## Steps 10 & 11: Running Locally & Deploying to the Cloud

### Run Locally with Docker Compose
```bash
unzip finpath-ai.zip
cd finpath-ai
cp .env.example .env
docker compose up --build
```
- Frontend: `http://localhost:3000`
- Backend API Docs: `http://localhost:8000/docs`

### Permanent Cloud Deployment
- Deploy `backend/Dockerfile` and `frontend/Dockerfile` to Render, Railway, AWS ECS, or Google Cloud Run.
- Set `JWT_SECRET_KEY`, `DATABASE_URL` (PostgreSQL or persistent volume SQLite), and `CORS_ORIGINS` in your cloud environment variables.